# 面向对象编程

**学习目标：** 本节介绍类、实例、构造函数、属性、方法、继承和方法重写，目标是掌握面向对象建模的基本结构。

`Python` 从设计之初就是一门面向对象语言。在 `Python` 中创建类和对象很直接，本章会围绕 `Python` 的面向对象编程展开。

如果你以前没有接触过面向对象编程，需要先理解几个基本概念：类负责抽象对象的共同特征，对象是类的具体实例，方法描述对象可以执行的行为。理解这些概念后，再学习 `Python` 的类语法会更清晰。


## 面向对象简介

- **类：** 用来描述具有相同属性和方法的一组对象。对象是类的实例。
- **对象：** 由类创建的数据结构实例，通常包含属性和方法。
- **实例化：** 根据类创建具体对象的过程。
- **属性：** 保存对象状态的数据，可以分为类属性和实例属性。
- **方法：** 定义在类中的函数，用来描述对象行为。
- **继承：** 子类复用父类的属性和方法，适合表达 `is-a` 关系。例如，`Dog` 可以继承 `Animal`。
- **方法重写：** 子类重新实现从父类继承的方法，以满足自身需求。


和其它编程语言相比，`Python` 在尽可能不增加新的语法和语义的情况下加入了类机制。

`Python` 中的类提供了面向对象编程的所有基本功能：类的继承机制允许多个基类，派生类可以覆盖基类中的任何方法，方法中可以调用基类中的同名方法。

对象可以包含任意数量和类型的数据。


## 类定义


In [ ]:
class Person:
    def set_name(self, name):
        self.name = name

    def print_name(self):
        print(self.name)


`class` 后直接跟类名，类名通常以大写字母开头，并且采用单数形式。

对于新式类，一般会在类名后加括号并继承 `object`。如果没有更具体的父类，可以继承 `object`，例如：

```python
class Person:
    pass
```


类实例化后，可以使用其属性，实际上，创建一个类之后，可以通过类名访问其属性。


类对象支持两种操作：**属性引用和实例化**。

属性引用使用和 `Python` 中所有的属性引用一样的标准语法：`obj.name`。

类对象创建后，类命名空间中所有的命名都是有效属性名。


In [ ]:
person = Person()  # 创建实例

# 调用实例方法
person.set_name("yemengjie")
person.print_name()

# 引用实例属性
print(person.name)


### `self` 代表类的实例，而非类


类方法与普通函数只有一个关键区别：实例方法必须有一个额外的第一个参数，按照惯例命名为 `self`。

> 实例方法的第一个参数 `self` 表示实例本身。调用实例方法时，不需要手动传入 `self`，解释器会自动传入当前实例。


## 构造函数与资源管理


在 `Python` 中，构造函数通常由特殊方法 `__init__()` 实现。创建类实例时，`Python` 会自动调用 `__init__()` 初始化实例状态。


`__init__()` 方法的第一个参数永远是 `self`，表示创建的实例本身，因此，在 `__init__()` 方法内部，就可以将各种属性绑定到 `self`，因为 `self` 就指向创建的实例本身。


In [ ]:
class Person:
    def __init__(self, name):
        self.name = name

    def print_name(self):
        print(self.name)


In [ ]:
# 传入 name
person = Person("yemengjie")
person.print_name()


`Python 3` 创建实例时通常涉及两个特殊方法：`__new__()` 和 `__init__()`。调用顺序是先执行 `__new__()`，再执行 `__init__()`。

- `__new__()` 负责创建并返回实例对象，第一个参数是类本身 `cls`。
- `__init__()` 负责初始化实例对象，第一个参数是实例本身 `self`，通常不显式返回值。

大多数普通类只需要实现 `__init__()`。只有在继承不可变类型、控制实例创建过程或实现单例等特殊场景中，才需要自定义 `__new__()`。

> 如果 `__new__()` 返回的不是当前类或其子类的实例，`__init__()` 不会继续执行。


In [ ]:
class Person:
    """演示 `__new__()` 和 `__init__()` 的调用顺序。"""

    def __new__(cls, name: str):
        """创建并返回当前类的实例。"""
        print("call __new__")
        return super().__new__(cls)

    def __init__(self, name: str) -> None:
        """初始化实例属性。"""
        print("call __init__")
        self.name = name

    def print_name(self) -> None:
        """打印实例名称。"""
        print(self.name)


person = Person("yemengjie")
person.print_name()


### 资源管理：优先使用上下文管理器

`__del__()` 是对象即将被销毁时可能调用的特殊方法。它的调用时机不确定，可能被延迟到垃圾回收或解释器关闭阶段，因此不适合用来管理关键资源。

在现代 `Python` 代码中，文件、网络连接、锁等资源更推荐使用 `with` 语句和上下文管理器管理。如果确实需要释放资源，应优先实现上下文管理协议 `__enter__()` 和 `__exit__()`，或者使用标准库 `contextlib`。


In [ ]:
class ManagedResource:
    """演示上下文管理器的资源获取与释放。"""

    def __enter__(self) -> "ManagedResource":
        print("open resource")
        return self

    def __exit__(self, exc_type, exc, traceback) -> bool:
        print("close resource")
        return False

    def use(self) -> None:
        print("use resource")


with ManagedResource() as resource:
    resource.use()


## 属性、函数和方法


- **实例属性**
> 属于某个对象或实例的属性
- **类属性**
> 不独属于某个实例，而是由所有实例共享的属性。


In [ ]:
class Person:
    """演示类属性和实例属性。"""

    # 类属性由所有实例共享。
    species = "human"

    def __init__(self, name="no one"):
        """初始化实例属性。"""
        self.name = name

p1 = Person()
print(p1.species)
print(p1.name)

p2 = Person("yemengjie")
print(p2.species)
print(p2.name)


- **类方法**
  类对象拥有的方法，使用 `@classmethod` 标识。类方法的第一个参数通常命名为 `cls`，可以通过类或实例访问。
- **实例方法**
  实例对象拥有的方法，第一个参数通常命名为 `self`。
- **静态方法**
  与类和实例状态都无关的方法，使用 `@staticmethod` 标识。


In [ ]:
class Person:
    """演示实例方法、类方法和静态方法。"""

    species = "human"

    def __init__(self, name="no one"):
        """初始化实例属性。"""
        self.name = name

    def instance_method(self):
        """实例方法可以访问实例属性。"""
        print("instance method", self.name)

    @classmethod
    def class_method(cls):
        """类方法可以访问类属性。"""
        print("class method", cls.species)

    @staticmethod
    def static_method():
        """静态方法不依赖类或实例状态。"""
        print("static method", Person.species)


In [ ]:
person = Person()
person.instance_method()

Person.class_method()
Person.static_method()

# 建议通过类调用
person.class_method()
person.static_method()


- `@staticmethod` 不需要类或实例信息，`@classmethod` 需要类信息，普通实例方法需要实例信息。
- 使用普通实例方法前，一般需要先实例化对象。
- 使用 `@staticmethod` 或 `@classmethod` 时，可以直接通过 `ClassName.method_name()` 调用。
- `@staticmethod` 适合组织与类相关，但不依赖类状态或实例状态的函数。
- `@classmethod` 常用于工厂方法，也就是 alternative constructor。它适合在创建真实实例之前做预处理，同时可以降低构造函数变更带来的影响。


### 私有成员

`__private_attrs`：两个下划线开头，表示私有属性。私有属性不应在类外部直接访问，在类内部通常通过 `self.__private_attrs` 使用。

`__private_method`：两个下划线开头，表示私有方法。私有方法只应在类内部调用，在类内部通常通过 `self.__private_method()` 使用。

### 受保护成员

`_protected_attrs`：一个下划线开头，表示受保护属性。受保护属性约定只在类或子类内部使用。

`_protected_method`：一个下划线开头，表示受保护方法。受保护方法约定只在类或子类内部调用。


In [ ]:
class Person:
    """演示公开属性、私有属性和受保护属性。"""

    species = "human"

    def __init__(self, name="no one", age=18, address="bnuz"):
        """初始化不同访问约定的实例属性。"""
        self.name = name  # 公开属性。
        self.__age = age  # 私有属性。
        self._address = address  # 受保护属性。

    def show_profile(self):
        """打印实例的基本信息。"""
        print(f"name: {self.name}, age: {self.__age}, address: {self._address}")


In [ ]:
person = Person()
person.show_profile()
print(person.name)
print(person._address)
print(person.__age)


## 扩展类功能


常见特殊方法包括：

| 方法 | 作用 |
| :--- | :--- |
| `__init__()` | 初始化实例。 |
| `__str__()` | 返回面向用户的字符串表示，供 `str()` 和 `print()` 使用。 |
| `__repr__()` | 返回面向开发者的字符串表示，供 `repr()` 和调试使用。 |
| `__setitem__()` | 支持 `obj[key] = value`。 |
| `__getitem__()` | 支持 `obj[key]`。 |
| `__len__()` | 支持 `len(obj)`。 |
| `__call__()` | 支持把实例当函数调用。 |
| `__eq__()` | 支持 `==`。 |
| `__lt__()` | 支持 `<`。 |
| `__add__()` | 支持 `+`。 |
| `__sub__()` | 支持 `-`。 |
| `__mul__()` | 支持 `*`。 |
| `__truediv__()` | 支持 `/`。 |
| `__floordiv__()` | 支持 `//`。 |
| `__mod__()` | 支持 `%`。 |
| `__pow__()` | 支持 `**`。 |

> `Python 3` 使用富比较方法和新的除法协议。比较运算应实现 `__eq__()`、`__lt__()` 等方法；除法运算应实现 `__truediv__()` 或 `__floordiv__()`。


### 格式化实例


要改变一个实例的字符串表示，可重新定义它的 `__str__()` 和 `__repr__()` 方法。例如：


In [ ]:
class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __repr__(self):
        return f'Vector({self.x}, {self.y})'

    def __str__(self):
        return f'({self.x}, {self.y})'


In [ ]:
v = Vector(3, 4)


In [ ]:
v


In [ ]:
print(v)


`__repr__()` 方法返回一个实例的代码表示形式，通常用来重新构造这个实例。内置的 `repr()` 函数返回这个字符串，跟我们使用交互式解释器显示的值是一样的。`__str__()` 方法将实例转换为一个字符串，使用 `str()` 或 `print()` 函数会输出这个字符串。

如果 `__str__()` 没有被定义，那么就会使用 `__repr__()` 来代替输出。


### 运算符重载

`Python` 支持运算符重载。通过实现类的特殊方法，可以让自定义对象支持 `+`、`-`、`==` 等常见运算符。示例如下：


In [ ]:
class Vector:
    """二维向量，演示特殊方法和运算符重载。"""

    def __init__(self, x, y):
        """初始化向量坐标。"""
        self.x = x
        self.y = y

    def __repr__(self):
        """返回适合调试的字符串表示。"""
        return f"Vector({self.x}, {self.y})"

    def __str__(self):
        """返回适合用户阅读的字符串表示。"""
        return f"({self.x}, {self.y})"

    def __add__(self, other):
        """返回两个向量相加后的新向量。"""
        return Vector(self.x + other.x, self.y + other.y)

    def __sub__(self, other):
        """返回两个向量相减后的新向量。"""
        return Vector(self.x - other.x, self.y - other.y)

    def __len__(self):
        """返回向量长度平方的整数值，用于演示比较逻辑。"""
        return int(self.x ** 2 + self.y ** 2)

    def __eq__(self, other):
        """比较两个向量长度平方是否相等。"""
        return len(self) == len(other)

    def __lt__(self, other):
        """比较当前向量是否短于另一个向量。"""
        return len(self) < len(other)


In [ ]:
v1 = Vector(1, 1)
v2 = Vector(2, 2)
print(len(v2))
print(f"{v1} + {v2} = {v1 + v2}")
print(f"{v1} - {v2} = {v1 - v2}")


In [ ]:
v1 >= v2


## 继承

`Python` 支持类继承。继承可以让子类复用父类的属性和方法，也可以让子类在父类基础上扩展新行为。派生类的基本结构如下：

```python
class DerivedClassName(BaseClassName):
    statement_1
    statement_2
```


需要注意圆括号中基类的顺序。如果多个基类中存在同名方法，而子类调用时没有明确指定，`Python` 会按从左到右的顺序查找：先查找子类，再按顺序查找父类。

示例中的基类名 `BaseClassName` 必须与派生类定义在同一个作用域内。除了类名，也可以使用表达式作为基类；当基类定义在另一个模块中时，这种写法很有用。


In [ ]:
class Person:
    """父类：保存人的基础信息。"""

    def __init__(self, name, age, weight):
        """初始化姓名、年龄和体重。"""
        self.name = name
        self.age = age
        self.__weight = weight

    def show_basic_info(self):
        """打印基础信息。"""
        print(f"{self.name}: age={self.age}, weight={self.__weight} kg")

# 单继承示例。
class Student(Person):
    """子类：在人的基础信息上增加年级。"""

    def __init__(self, name, age, weight, grade):
        """先初始化父类属性，再初始化子类属性。"""
        Person.__init__(self, name, age, weight)
        self.grade = grade

    def show_student_info(self):
        """打印学生信息。"""
        print(f"{self.name}: age={self.age}, grade={self.grade}")

student = Student("ken", 10, 30, 3)
student.show_basic_info()
student.show_student_info()


## 多继承

`Python` 支持多继承，但多继承会增加方法解析顺序和状态初始化的复杂度。实际项目中应谨慎使用，优先选择组合或更清晰的继承结构。

多继承的基本结构如下：

```python
class DerivedClassName(Base1, Base2, Base3):
    statement_1
    statement_2
```


多继承时需要注意父类顺序。如果多个父类存在同名方法，而子类没有重写该方法，`Python` 会按照方法解析顺序 `MRO` 查找方法。可以使用 `ClassName.mro()` 查看具体查找顺序。


In [ ]:
class Person:
    """父类：保存人的基础信息。"""

    def __init__(self, name, age, weight):
        """初始化人的基础信息。"""
        print("init Person")
        self.name = name
        self.age = age
        self.__weight = weight

    def show_basic_info(self):
        """打印基础信息。"""
        print(f"{self.name}: age={self.age}, weight={self.__weight} kg")

class Student(Person):
    """学生类：继承人的基础信息，并增加年级。"""

    def __init__(self, name, age, weight, grade):
        """初始化学生信息。"""
        print("init Student")
        Person.__init__(self, name, age, weight)
        self.grade = grade

    def show_student_info(self):
        """打印学生信息。"""
        print(f"{self.name}: age={self.age}, grade={self.grade}")

class Speaker:
    """演讲者类：保存演讲主题。"""

    def __init__(self, name, topic):
        """初始化演讲者信息。"""
        print("init Speaker")
        self.name = name
        self.topic = topic

    def speak(self):
        """打印演讲主题。"""
        print(f"I'm {self.name}, my topic is {self.topic}")

class Sample(Student, Speaker):
    """多继承示例类。"""

    def __init__(self, name, age, weight, grade, topic):
        """分别初始化两个父类。"""
        print("init Sample")
        Student.__init__(self, name, age, weight, grade)
        Speaker.__init__(self, name, topic)

sample = Sample("Tim", 25, 80, 4, "Python")
# 多继承中如果父类方法同名，会按照方法解析顺序查找。
sample.speak()


实际上，考虑到可维护性，不是很建议使用多继承，更多的情况下使用组合


In [ ]:
class Person:
    """父类：保存人的基础信息。"""

    def __init__(self, name, age, weight):
        """初始化人的基础信息。"""
        self.name = name
        self.age = age
        self.__weight = weight

    def speak(self):
        """打印自我介绍。"""
        print(f"{self.name}: I'm {self.age} years old")

class Speaker(Person):
    """演讲者类：继承人的基础信息，并增加演讲主题。"""

    def __init__(self, name, age, weight, topic):
        """初始化演讲者信息。"""
        Person.__init__(self, name, age, weight)
        self.topic = topic

    def speak(self):
        """打印演讲者信息。"""
        print(f"I'm {self.name}, my topic is {self.topic}")

# 组合 Speaker
class Sample:
    """组合示例类。"""

    def __init__(self, name, age, weight, topic):
        """保存一个 Speaker 实例。"""
        self.speaker = Speaker(name, age, weight, topic)

test = Sample("Tim", 30, 60, "Python")
test.speaker.speak()
print(f"name={test.speaker.name}, age={test.speaker.age}, topic={test.speaker.topic}")


## 方法重写

如果父类方法不能满足子类需求，可以在子类中定义同名方法覆盖父类实现。示例如下：


In [ ]:
class Parent:
    """父类示例。"""

    def my_method(self):
        """打印父类方法信息。"""
        print("调用父类方法")

class Child(Parent):
    """子类示例。"""

    def my_method(self):
        """重写父类方法。"""
        print("调用子类方法")

child = Child()
child.my_method()
super(Child, child).my_method()


`super()` 用于调用父类或方法解析顺序 `MRO` 中的下一个类的方法。它常用于子类扩展父类逻辑，而不是完全重写父类逻辑。


### 子类构造函数与父类构造函数

子类定义自己的 `__init__()` 后，父类的 `__init__()` 不会自动执行。此时需要在子类构造函数中显式调用父类构造函数，常见方式是使用 `super()`。


如果在子类中需要父类的构造方法就需要显式的调用父类的构造方法，或者不重写父类的构造方法。

子类不重写 `__init__`，实例化子类时，会自动调用父类定义的 `__init__`。


In [ ]:
class Father(object):
    """父类：定义姓名和获取姓名的方法。"""

    def __init__(self):
        """初始化父类姓名。"""
        self.name = "mxh"
        print(f"name: {self.name}")

    def get_name(self):
        """返回父类格式的姓名。"""
        return f"Father {self.name}"

class Son(Father):
    """子类：重写获取姓名的方法。"""

    def get_name(self):
        """返回子类格式的姓名。"""
        return f"Son {self.name}"

if __name__ == "__main__":
    son = Son()
    print(son.get_name())


如果重写了 `__init__` 时，实例化子类，就不会调用父类已经定义的 `__init__`，语法格式如下：


In [ ]:
class Father(object):
    """父类：定义姓名和获取姓名的方法。"""

    def __init__(self, name):
        """初始化父类姓名。"""
        self.name = name
        print(f"name: {self.name}")

    def get_name(self):
        """返回父类格式的姓名。"""
        return f"Father {self.name}"

class Son(Father):
    """子类：定义自己的构造函数。"""

    def __init__(self, name):
        """直接初始化子类姓名，不调用父类构造函数。"""
        print("hi")
        self.name = name

    def get_name(self):
        """返回子类格式的姓名。"""
        return f"Son {self.name}"

son = Son("morningstar")
print(son.get_name())


重写 `__init__()` 时，如果还需要执行父类初始化逻辑，推荐使用 `super()`：

```python
super().__init__(arg1, arg2)
```

也可以显式调用父类方法，但这种写法耦合更强，通常不如 `super()` 灵活：

```python
BaseClass.__init__(self, arg1, arg2)
```


In [ ]:
class Father(object):
    """父类：定义姓名和获取姓名的方法。"""

    def __init__(self, name):
        """初始化父类姓名。"""
        self.name = name
        print(f"father name: {self.name}")

    def get_name(self):
        """返回父类格式的姓名。"""
        return f"Father {self.name}"

class Son(Father):
    """子类：通过 `super()` 调用父类构造函数。"""

    def __init__(self, name):
        """先调用父类构造函数，再补充子类初始化逻辑。"""
        print(type(super(Son, self)))
        super().__init__(name)
        print("Son init")

    def get_name(self):
        """返回子类格式的姓名。"""
        return f"Son {self.name}"

if __name__ == "__main__":
    son = Son("morningstar")
    print(son.get_name())


## 练习


## 三角形类练习

1. 创建一个 `Triangle` 类。其 `__init__()` 方法接收 `angle1`、`angle2` 和 `angle3`，表示三角形的三个内角。
2. 创建一个名为 `number_of_sides` 的变量，并将其设置为 `3`。
3. 创建一个名为 `check_angles()` 的方法。如果三个角的和等于 `180`，返回 `True`，否则返回 `False`。
4. 创建一个名为 `my_triangle` 的变量，并将其设置为 `Triangle` 的实例，例如 `Triangle(90, 30, 60)`。
5. 打印 `my_triangle.number_of_sides`，并调用 `my_triangle.check_angles()` 检查三角形内角和是否符合条件。


### 第 2 题

1. 定义一个名为 `Song` 的类，用来显示一首歌的歌词。`__init__()` 方法接收一个参数 `lyrics`，其中 `lyrics` 是一个列表。
2. 在类中创建一个名为 `sing_me_a_song()` 的方法，逐行打印 `lyrics` 中的每个元素。
3. 定义变量：

   ```python
   happy_bday = Song(["May god bless you, ", "Have a sunshine on you,", "Happy Birthday to you!"])
   ```

4. 在 `happy_bday` 上调用 `sing_me_a_song()` 方法。


### 第 3 题

1. 定义一个 `Point3D` 类。
2. 定义 `__init__()` 方法，接收 `x`、`y` 和 `z`，并将它们分别保存到 `self.x`、`self.y` 和 `self.z`。
3. 定义 `__repr__()` 方法，返回 `(x, y, z)` 形式的字符串。
4. 在类定义之外创建变量 `my_point`，保存一个新的 `Point3D` 实例，其中 `x=1`、`y=2`、`z=3`。
5. 打印 `my_point`。
